Celda 1: Montar Drive e instalar bert-score y librerías de visualización

In [ ]:
import os, random
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/paperizar'
os.chdir(BASE_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

!pip install -q bert-score rouge-score textstat spacy scipy pandas matplotlib mlflow tqdm pyyaml
print("Entorno listo. GPU disponible para BERTScore:", torch.cuda.is_available())

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Entorno listo. GPU disponible para BERTScore: True


Celda 2: Crear src/paperizar/verify.py (Módulo verificador de pares y generador del Reporte JSON auditable)Calcula los percentiles 10 y 90 de Flesch sobre corpus_A (solo train, tal como exige la Sección 5-F), detecta entidades perdidas (entities.lost), añadidas (entities.added), cambios de unidad sobre una misma cifra (units_changed, ej. 12 ms $\rightarrow$ 12 s) y redacta las advertencias legibles (warnings) exigidas en la página 18.

In [ ]:
%%writefile src/paperizar/verify.py
import os, re, json
from typing import Dict, List, Tuple
import numpy as np
import spacy
import textstat
from src.paperizar.informalize import entity_present, norm_ws, clean_measure

UNIT_SPLIT_RE = re.compile(r'^(\d+(?:\.\d+)?)\s*(%|million|billion|thousand|[A-Za-z]+)$')

def compute_corpus_flesch_bounds(
    corpus_path: str = "data/processed/corpus_A.jsonl",
    cache_path: str = "configs/flesch_bounds.json"
) -> Tuple[float, float]:
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            d = json.load(f)
        return float(d["p10"]), float(d["p90"])

    scores = []
    with open(corpus_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if r.get("split") == "train":
                scores.append(textstat.flesch_reading_ease(r["A"]))

    p10, p90 = float(np.percentile(scores, 10)), float(np.percentile(scores, 90))
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump({"p10": round(p10, 4), "p90": round(p90, 4), "n_train": len(scores)}, f, indent=2)
    return p10, p90

def parse_num_unit(measure_txt: str) -> Tuple[str, str]:
    c = clean_measure(measure_txt)
    m = UNIT_SPLIT_RE.match(c)
    if m:
        return m.group(1), m.group(2)
    return "", ""

def detect_units_changed(ents_in: List[Dict], ents_out: List[Dict]) -> List[Dict]:
    in_map = {}
    for e in ents_in:
        if e["label"] == "MEASURE":
            num, unit = parse_num_unit(e["text"])
            if num and unit:
                in_map.setdefault(num, set()).add(unit)

    changes = []
    for e in ents_out:
        if e["label"] == "MEASURE":
            num, unit_out = parse_num_unit(e["text"])
            if num and unit_out and num in in_map:
                if unit_out not in in_map[num]:
                    changes.append({
                        "number": num,
                        "input_units": sorted(in_map[num]),
                        "output_unit": unit_out,
                        "span": e["text"]
                    })
    return changes

def verify_single_rewrite(
    input_text: str,
    output_text: str,
    nlp: spacy.Language,
    flesch_bounds: Tuple[float, float]
) -> Dict:
    """Genera el reporte JSON oficial de verificación (Sección 7 / Página 18)."""
    doc_in = nlp(input_text)
    doc_out = nlp(output_text)
    ents_in = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_in.ents]
    ents_out = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_out.ents]

    lost = [e["text"] for e in ents_in if not entity_present(e, ents_out, output_text)]
    added = [e["text"] for e in ents_out if not entity_present(e, ents_in, input_text)]
    u_changed = detect_units_changed(ents_in, ents_out)

    fl_in = round(float(textstat.flesch_reading_ease(input_text)), 2)
    fl_out = round(float(textstat.flesch_reading_ease(output_text)), 2)
    p10, p90 = flesch_bounds
    in_range = bool(p10 <= fl_out <= p90)

    warnings = []
    if lost:
        warnings.append(f"Se omitieron {len(lost)} entidades técnicas de la entrada: {', '.join(lost[:5])}.")
    if added:
        warnings.append(f"Aparecen {len(added)} entidades nuevas no detectadas en la entrada (posible expansión de siglas o alucinación): {', '.join(added[:5])}.")
    if u_changed:
        desc = ", ".join(f"{c['number']} ({'/'.join(c['input_units'])} -> {c['output_unit']})" for c in u_changed)
        warnings.append(f"Alerta crítica de unidades alteradas: {desc}.")
    if fl_out > fl_in:
        warnings.append(f"El texto de salida no aumentó su complejidad académica (Flesch subió de {fl_in} a {fl_out}).")
    if not in_range:
        warnings.append(f"Legibilidad Flesch ({fl_out}) fuera del rango típico [p10={p10:.1f}, p90={p90:.1f}] de abstracts de ingeniería.")

    return {
        "output": output_text,
        "entities": {
            "lost": lost,
            "added": added
        },
        "units_changed": u_changed,
        "flesch": {
            "input": fl_in,
            "output": fl_out,
            "reference_p10_p90": [round(p10, 2), round(p90, 2)],
            "in_range": in_range
        },
        "warnings": warnings
    }

def compute_pair_entity_metrics(
    text_a: str,
    text_i: str,
    text_aprime: str,
    nlp: spacy.Language,
    flesch_bounds: Tuple[float, float]
) -> Dict:
    doc_a = nlp(text_a)
    doc_i = nlp(text_i)
    doc_ap = nlp(text_aprime)

    ents_a = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_a.ents]
    ents_i = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_i.ents]
    ents_ap = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_ap.ents]

    ent_cov_A = sum(1 for e in ents_a if entity_present(e, ents_ap, text_aprime)) / len(ents_a) if ents_a else 1.0
    ent_prec_A = sum(1 for e in ents_ap if entity_present(e, ents_a, text_a)) / len(ents_ap) if ents_ap else 1.0
    ent_cov_I = sum(1 for e in ents_i if entity_present(e, ents_ap, text_aprime)) / len(ents_i) if ents_i else 1.0

    # unit_preserved: todas las cifras con unidad de I (o A presentes en I) se conservan idénticas sin cambio de unidad
    measures_ref = [e for e in ents_a if e["label"] == "MEASURE" and entity_present(e, ents_i, text_i)]
    all_meas_kept = all(entity_present(m, ents_ap, text_aprime) for m in measures_ref)
    no_unit_swap = (len(detect_units_changed(ents_i, ents_ap)) == 0)
    unit_preserved = bool(all_meas_kept and no_unit_swap)

    fl_a = float(textstat.flesch_reading_ease(text_a))
    fl_i = float(textstat.flesch_reading_ease(text_i))
    fl_ap = float(textstat.flesch_reading_ease(text_aprime))
    p10, p90 = flesch_bounds

    return {
        "ent_cov_A": round(ent_cov_A, 4),
        "ent_prec_A": round(ent_prec_A, 4),
        "ent_cov_I": round(ent_cov_I, 4),
        "unit_preserved": unit_preserved,
        "flesch_A": round(fl_a, 2),
        "flesch_I": round(fl_i, 2),
        "flesch_Aprime": round(fl_ap, 2),
        "flesch_harder_than_I": bool(fl_ap < fl_i),
        "flesch_in_range": bool(p10 <= fl_ap <= p90)
    }

Writing src/paperizar/verify.py


Celda 3: Crear src/paperizar/evaluate.py (Pasos P5 a P9: BERTScore, Controles, Bootstrap 10.000, Wilcoxon + Holm y Muestra Ciega de 40 Pares)Implementa exactamente la Sección 4 y la Sección 5-F: incluye el control de identidad ($A' = I$), BERTScore con rescale_with_baseline=True, intervalos de confianza del 95% con 10.000 remuestreos bootstrap (semilla 42), prueba pareada de Wilcoxon con corrección de Holm y la exportación de la hoja ciega estratificada de 40 pares para calificación humana.

In [ ]:
%%writefile src/paperizar/evaluate.py
import os, json, random
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import spacy
from scipy.stats import wilcoxon
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm
from src.paperizar.verify import compute_corpus_flesch_bounds, compute_pair_entity_metrics

SEED = 42

def bootstrap_ci(values: np.ndarray, n_boot: int = 10000, seed: int = 42) -> Tuple[float, float, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(arr), size=(n_boot, len(arr)))
    means = arr[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(arr.mean()), float(lo), float(hi)

def holm_bonferroni(p_values: List[float]) -> List[float]:
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.zeros(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = p_values[idx] * (m - rank)
        running_max = max(running_max, min(1.0, adj))
        adjusted[idx] = running_max
    return adjusted.tolist()

def evaluate_all_systems(
    paperized_jsonl: str = "data/processed/paperized_roundtrip.jsonl",
    out_roundtrip_jsonl: str = "results/roundtrip.jsonl",
    strict_keep_only: bool = True
) -> pd.DataFrame:
    nlp = spacy.load("models/ner_hybrid_v1")
    flesch_bounds = compute_corpus_flesch_bounds()
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    with open(paperized_jsonl, encoding="utf-8") as f:
        raw_rows = [json.loads(line) for line in f if line.strip()]

    if strict_keep_only:
        rows = [r for r in raw_rows if not r.get("fallback_100_coverage", False)]
    else:
        rows = raw_rows

    # Añadir control de identidad (A' = I) exigido en Paso P7 una vez por borrador único
    seen_drafts = {}
    for r in rows:
        if r["id"] not in seen_drafts:
            ctrl = dict(r)
            ctrl["A_prime"] = r["I"]
            ctrl["model_id"] = "control_identity"
            ctrl["prompt_id"] = "CTRL-IDENTITY"
            ctrl["prompt_hash"] = "000000000000"
            seen_drafts[r["id"]] = ctrl

    all_eval_rows = rows + list(seen_drafts.values())
    print(f"Calculando BERTScore sobre {len(all_eval_rows)} salidas (Herramienta + Línea Base + Control Identidad)...")

    cands = [r["A_prime"] for r in all_eval_rows]
    refs = [r["A"] for r in all_eval_rows]
    _, _, F1_bert = bert_score_fn(
        cands,
        refs,
        lang="en",
        rescale_with_baseline=True,
        batch_size=32,
        verbose=True
    )
    f1_list = F1_bert.tolist()

    enriched = []
    for r, bf1 in tqdm(zip(all_eval_rows, f1_list), total=len(all_eval_rows), desc="Métricas NER + Flesch + ROUGE"):
        rl = rouge.score(r["A"], r["A_prime"])["rougeL"].fmeasure
        m_dict = compute_pair_entity_metrics(r["A"], r["I"], r["A_prime"], nlp, flesch_bounds)
        rec_out = {
            **r,
            "bertscore_f1": round(float(bf1), 4),
            "rougeL_f1": round(float(rl), 4),
            **m_dict
        }
        enriched.append(rec_out)

    os.makedirs(os.path.dirname(out_roundtrip_jsonl), exist_ok=True)
    with open(out_roundtrip_jsonl, "w", encoding="utf-8") as fout:
        for r in enriched:
            fout.write(json.dumps(r, ensure_ascii=False) + "\n")

    return pd.DataFrame(enriched)

def run_statistical_tests(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metrics = [
        "ent_cov_A", "ent_prec_A", "ent_cov_I", "unit_preserved",
        "bertscore_f1", "rougeL_f1", "flesch_Aprime", "flesch_in_range", "flesch_harder_than_I"
    ]

    # 1. Resumen global por sistema con IC 95% Bootstrap (10.000 remuestreos)
    sys_rows = []
    for sys_id, grp in df.groupby("prompt_id"):
        row = {"Sistema": sys_id, "N_pares": len(grp)}
        for m in metrics:
            mean_v, lo, hi = bootstrap_ci(grp[m].astype(float).values, n_boot=10000, seed=SEED)
            row[f"{m}_mean"] = round(mean_v, 4)
            row[f"{m}_ci95"] = f"[{lo:.4f}, {hi:.4f}]"
        sys_rows.append(row)
    df_sys = pd.DataFrame(sys_rows)

    # 2. Comparación pareada Herramienta (P-PAP-v1) vs Línea Base (P-BASE-v1) con Wilcoxon + Holm (Paso P9)
    piv_pap = df[df["prompt_id"] == "P-PAP-v1"].set_index("id")
    piv_bas = df[df["prompt_id"] == "P-BASE-v1"].set_index("id")
    common_ids = piv_pap.index.intersection(piv_bas.index)
    piv_pap = piv_pap.loc[common_ids]
    piv_bas = piv_bas.loc[common_ids]

    test_metrics = ["ent_cov_A", "ent_prec_A", "unit_preserved", "bertscore_f1", "rougeL_f1", "flesch_in_range"]
    raw_pvals, comp_rows = [], []

    for m in test_metrics:
        diff = piv_pap[m].astype(float).values - piv_bas[m].astype(float).values
        d_mean, d_lo, d_hi = bootstrap_ci(diff, n_boot=10000, seed=SEED)
        if np.allclose(diff, 0):
            stat_w, p_val = 0.0, 1.0
        else:
            res_w = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
            stat_w, p_val = float(res_w.statistic), float(res_w.pvalue)
        raw_pvals.append(p_val)
        comp_rows.append({
            "Metrica": m,
            "Media_Herramienta": round(float(piv_pap[m].astype(float).mean()), 4),
            "Media_LineaBase": round(float(piv_bas[m].astype(float).mean()), 4),
            "Dif_Pareada_Media": round(d_mean, 4),
            "Dif_IC95_Bootstrap": f"[{d_lo:.4f}, {d_hi:.4f}]",
            "Wilcoxon_stat": round(stat_w, 2),
            "p_valor_crudo": p_val
        })

    adj_pvals = holm_bonferroni(raw_pvals)
    for r, p_adj in zip(comp_rows, adj_pvals):
        r["p_valor_Holm"] = round(p_adj, 6)
        r["Significativo_0.05"] = bool(p_adj < 0.05)

    return df_sys, pd.DataFrame(comp_rows)

def export_blind_human_sheet(df: pd.DataFrame, out_csv: str = "results/hoja_revision_humana_40_ciega.csv", key_json: str = "results/llave_revision_humana_40.json"):
    """Paso P6: 40 pares estratificados por generador y nivel, a ciegas y en orden aleatorio."""
    rng = random.Random(SEED)
    pap = df[df["prompt_id"] == "P-PAP-v1"].set_index("id")
    bas = df[df["prompt_id"] == "P-BASE-v1"].set_index("id")
    common = list(pap.index.intersection(bas.index))

    by_strata = {}
    for pid in common:
        g = pap.loc[pid, "generator"]
        l = int(pap.loc[pid, "level"])
        by_strata.setdefault((g, l), []).append(pid)

    chosen_ids = []
    strata_keys = sorted(by_strata.keys())
    for sk in strata_keys:
        pool = by_strata[sk][:]
        rng.shuffle(pool)
        chosen_ids.extend(pool[:4])  # 9 estratos x 4 = 36

    leftovers = [pid for pid in common if pid not in chosen_ids]
    rng.shuffle(leftovers)
    chosen_ids.extend(leftovers[:max(0, 40 - len(chosen_ids))])
    chosen_ids = chosen_ids[:40]

    blind_rows, secret_key = [], []
    for idx, pid in enumerate(chosen_ids, 1):
        # Asignar aleatoriamente cuál es Texto_X y cuál es Texto_Y para cegar al revisor
        flip = rng.choice([True, False])
        sys_x, sys_y = ("P-PAP-v1", "P-BASE-v1") if not flip else ("P-BASE-v1", "P-PAP-v1")
        txt_x = pap.loc[pid, "A_prime"] if sys_x == "P-PAP-v1" else bas.loc[pid, "A_prime"]
        txt_y = pap.loc[pid, "A_prime"] if sys_y == "P-PAP-v1" else bas.loc[pid, "A_prime"]

        blind_rows.append({
            "item_id": f"HUM_{idx:02d}",
            "original_A": pap.loc[pid, "A"],
            "draft_I": pap.loc[pid, "I"],
            "candidate_X": txt_x,
            "X_significado_1a5": "",
            "X_registro_1a5": "",
            "X_fluidez_1a5": "",
            "candidate_Y": txt_y,
            "Y_significado_1a5": "",
            "Y_registro_1a5": "",
            "Y_fluidez_1a5": ""
        })
        secret_key.append({"item_id": f"HUM_{idx:02d}", "pair_id": pid, "generator": pap.loc[pid, "generator"], "level": int(pap.loc[pid, "level"]), "candidate_X": sys_x, "candidate_Y": sys_y})

    pd.DataFrame(blind_rows).to_csv(out_csv, index=False)
    with open(key_json, "w", encoding="utf-8") as f:
        json.dump(secret_key, f, indent=2)

Writing src/paperizar/evaluate.py


Celda 4: Crear src/paperizar/cli.py (Tarea T8: Interfaz de Consola Lista para Demostración en Sustentación)

Permite correr PYTHONPATH=src python -m paperizar.cli --text "..." o --input borrador.txt --output reporte.json y devuelve con una sola orden el abstract académico y su reporte JSON completo de verificación.

In [ ]:
%%writefile src/paperizar/cli.py
import sys, json, argparse
import spacy
from src.paperizar.generate import PaperizerEngine
from src.paperizar.verify import compute_corpus_flesch_bounds, verify_single_rewrite

def main():
    parser = argparse.ArgumentParser(
        prog="python -m paperizar.cli",
        description="Paperización verificable de abstracts de ingeniería en inglés."
    )
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--text", type=str, help="Borrador informal en inglés entre comillas.")
    group.add_argument("--input", type=str, help="Ruta a un archivo .txt con el borrador informal.")
    parser.add_argument("--output", type=str, default=None, help="Ruta opcional para guardar el reporte JSON.")
    parser.add_argument("--config", type=str, default="configs/generacion.yaml", help="Configuración de generación.")
    parser.add_argument("--ner-model", type=str, default="models/ner_hybrid_v1", help="Ruta al modelo NER híbrido.")
    args = parser.parse_args()

    draft = args.text.strip() if args.text else open(args.input, encoding="utf-8").read().strip()
    if not draft:
        sys.exit("Error: El texto de entrada está vacío.")

    nlp = spacy.load(args.ner_model)
    flesch_bounds = compute_corpus_flesch_bounds()

    engine = PaperizerEngine(args.config)
    gen_out = engine.generate_batch([draft], ["cli_interactive"], mode="tool")[0]
    engine.close()

    report = verify_single_rewrite(draft, gen_out["A_prime"], nlp, flesch_bounds)
    report["meta"] = {
        "model_id": gen_out["model_id"],
        "prompt_id": gen_out["prompt_id"],
        "prompt_hash": gen_out["prompt_hash"]
    }

    formatted_json = json.dumps(report, indent=2, ensure_ascii=False)
    if args.output:
        with open(args.output, "w", encoding="utf-8") as f:
            f.write(formatted_json + "\n")
    print(formatted_json)

if __name__ == "__main__":
    main()

Writing src/paperizar/cli.py


Celda 5: Ejecutar Evaluación Completa, Gráficas, Tablas, MLflow y Prueba en Vivo de python -m paperizar.cli

In [ ]:
import json, importlib
import matplotlib.pyplot as plt
import mlflow
import src.paperizar.verify as ver_mod
import src.paperizar.evaluate as eval_mod
importlib.reload(ver_mod)
importlib.reload(eval_mod)

# 1. Evaluar todo el conjunto roundtrip y guardar results/roundtrip.jsonl
df_rt = eval_mod.evaluate_all_systems(
    paperized_jsonl="data/processed/paperized_roundtrip.jsonl",
    out_roundtrip_jsonl="results/roundtrip.jsonl",
    strict_keep_only=True
)

# 2. Tablas estadísticas (Bootstrap 10.000 + Wilcoxon con corrección de Holm)
df_sys, df_comp = eval_mod.run_statistical_tests(df_rt)
df_sys.to_csv("results/tabla_sistemas_global.csv", index=False)
df_comp.to_csv("results/tabla_wilcoxon_holm.csv", index=False)

# Desglose por generador y nivel para la herramienta
df_tool = df_rt[df_rt["prompt_id"] == "P-PAP-v1"]
df_by_gen_lvl = df_tool.groupby(["generator", "level"]).agg(
    n=("id", "count"),
    ent_cov_A=("ent_cov_A", lambda s: f"{s.mean():.3f} ± {s.std():.3f}"),
    ent_prec_A=("ent_prec_A", lambda s: f"{s.mean():.3f} ± {s.std():.3f}"),
    unit_preserved=("unit_preserved", "mean"),
    bertscore_f1=("bertscore_f1", lambda s: f"{s.mean():.3f} ± {s.std():.3f}"),
    flesch_in_range=("flesch_in_range", "mean")
).reset_index()
df_by_gen_lvl.to_csv("results/tabla_por_generador_nivel.csv", index=False)

# 3. Exportar hoja ciega de 40 pares para revisión humana (Paso P6)
eval_mod.export_blind_human_sheet(df_rt)

# 4. Generar figura comparativa lista para el informe PDF
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
order_sys = ["CTRL-IDENTITY", "P-BASE-v1", "P-PAP-v1"]
labels_sys = ["Control\n(A' = I)", "Línea Base\n(P-BASE-v1)", "Herramienta\n(P-PAP-v1)"]

for ax, col, title in zip(
    axes,
    ["ent_cov_A", "unit_preserved", "bertscore_f1"],
    ["Cobertura de Entidades vs A", "Unidades Exactas Preservadas", "BERTScore F1 (reescalado)"]
):
    means = [df_rt[df_rt["prompt_id"] == s][col].astype(float).mean() for s in order_sys]
    bars = ax.bar(labels_sys, means, color=["#9aa0a6", "#4285f4", "#0f9d58"], width=0.55)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.08)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    for b, v in zip(bars, means):
        ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9.5, fontweight="bold")

plt.tight_layout()
plt.savefig("results/figura_comparacion_principal.png", dpi=200)
plt.close()

# 5. Registrar corrida final en MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("paperizar_evaluation")
with mlflow.start_run(run_name="roundtrip_full_evaluation"):
    for _, r in df_comp.iterrows():
        mlflow.log_metric(f"tool_{r['Metrica']}", r["Media_Herramienta"])
        mlflow.log_metric(f"base_{r['Metrica']}", r["Media_LineaBase"])
        mlflow.log_metric(f"pval_holm_{r['Metrica']}", r["p_valor_Holm"])

print("\n=== COMPARACIÓN PAREADA WILCOXON + HOLM (HERRAMIENTA VS LÍNEA BASE) ===")
print(df_comp.to_string(index=False))

# 6. Prueba de aceptación de Tarea T8: ejecutar python -m paperizar.cli con una sola orden
print("\n=== PRUEBA EN VIVO DE INTERFAZ DE CONSOLA (TAREA 8) ===")
!PYTHONPATH=src python -m paperizar.cli --text "We built a YOLOv8 detector on a Raspberry Pi 4 that hits 94.2% F1-score at 28 fps while drawing only 4.5 W." --output results/ejemplo_reporte_cli.json

Calculando BERTScore sobre 1779 salidas (Herramienta + Línea Base + Control Identidad)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (8 outputs left out)

done in 123.62 seconds, 14.39 sentences/sec


Métricas NER + Flesch + ROUGE:   0%|          | 0/1779 [00:00<?, ?it/s]

2026/11/16 19:01:13 INFO mlflow.tracking.fluent: Experiment with name 'paperizar_evaluation' does not exist. Creating a new experiment.



=== COMPARACIÓN PAREADA WILCOXON + HOLM (HERRAMIENTA VS LÍNEA BASE) ===
        Metrica  Media_Herramienta  Media_LineaBase  Dif_Pareada_Media Dif_IC95_Bootstrap  Wilcoxon_stat  p_valor_crudo  p_valor_Holm  Significativo_0.05
      ent_cov_A             0.8743           0.8434             0.0309   [0.0209, 0.0411]        13231.5   3.715227e-08      0.000000                True
     ent_prec_A             0.9392           0.9281             0.0110   [0.0037, 0.0187]        20971.5   1.555616e-02      0.015556                True
 unit_preserved             0.4958           0.3980             0.0978   [0.0641, 0.1332]         1610.0   5.566420e-08      0.000000                True
   bertscore_f1             0.4183           0.3671             0.0512   [0.0463, 0.0559]        19076.0   2.319807e-61      0.000000                True
      rougeL_f1             0.4238           0.3954             0.0284   [0.0244, 0.0324]        35559.5   4.561469e-36      0.000000                True
fle

Prueba del goldtrue

In [2]:
import os, sys, subprocess

# 0. Re-hidratar entorno de Colab por si se reinició la sesión
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/paperizar'
os.chdir(BASE_DIR)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "mlflow", "bert-score", "rouge-score", "textstat",
    "transformers", "accelerate", "spacy", "pyyaml", "scipy", "pandas", "tqdm"
])

import re, json
import pandas as pd
import spacy
import mlflow
from src.paperizar.informalize import compute_entity_coverage, entity_present
from src.paperizar.generate import PaperizerEngine, run_paperization_pipeline
from src.paperizar.evaluate import evaluate_all_systems

# 1. Respaldo embebido con el contenido exacto de respuestas_gold_real_sheets.txt
RAW_TXT_FALLBACK = r"""[1] (rt_096 | 2609.37711)
The work intends to solve via neuromorphic computing the power consumption in devices like smartphones, wireless headphones and hearing aids when using audio denoising neural networks. Though SNNs can potentially solve the problem thanks to their high activation sparsity and low complecity, many SOTA SNN's won't perform inference to its fullest without hardware that supports a mixture of operations. We can solve this by converting SOTA audio denoising neural network Spiking-FullSubNet to a hardware friendly version, which means QAT and simplifying function activation can achieve $\approx28\times$ improvement in power consumption to 52.9nJ per 32ms audio frame when calculated for custom digital hardware in a 45nm process node. Then, a digital circuit that can perform inference of the heterogeneous compute load of Spiking-FullSubNet with a sparsity-aware flexible PE array is proposed, so we can validate this circuit on a PYNQ-Z1 FPGA and finally achieve a real-time factor of of 0.727 at 100MHz.

--------------------------------------------------

[2] (rt_063 | 2610.08688)
The work looks to accelerate transformer-based LMs on resource-constrained IoT devices, presenting a framework that targets compact LM's such as B-Ty, M-Bt, M-Lm, E-Lt and D-Bt, all selected for being diverse architecturally and use in edge inference scenarios. The presented flow derives lightweight instruction set extensions tailored to the non-obvious computational patterns of these models. Plus, the address generation stage of batch matrix multiplication accelerates by introducing a custom instruction, which improves performance (15.39--21.74%) with modest ASIC overheads of 6.79% in area and 2.33% in power. The employment of an optional compiler-directed loop unrolling strategy further enhances performance without incurring additional processor core hardware cost, since it trades increased code sice for overall reduced execution time. By evaluating the Synopsys trv32p3f RISC-V core speeds up inference up to 2.19x, and implementing FPGA on the AMD Zynq UltraScale+ ZCU102 shows a 32.94% area overhead at 75 MHz, whereas the ASIC implementation using the TSMC 28 nm library incurs a 21.08% area overhead while operating at 250 MHz.

--------------------------------------------------

[3] (rt_084 | 2611.05165)
Public mobile networks may present many problems during disasters, so deployable communication systems are required to support emergency operations. This is why the article proposes a mission-critical communications system supporing MCPTT, MCVideo and MCData over a private 5G standaloe network with NTN connectivity. A context-aware QPP mechanism might dynamically adapt network resource allocation according to the operational state of MCX. Extensions to the Open5GS PCF and SMF enable priority and pre-emption updates for existing flows without session re-establishment. This architecture can both support tactical edge deployment of service and media functions and, at the same time, retain centralized service control. Three configurations are considered when evaluating via experiments: terrestrial backhaul with centralized media processing, satellite backhaul with centralized media processing, and satellite backhaul with edge media processing. Main results are that tactical edge processing reduces communication delay and packet loss; under satellite backhaul, edge processing reduced the 95th-percentile MCPTT audio transport delay by 60.03 ms and packet loss by 9.42 percentage points, while achieving a 95th-percentile MCPTT access time of 184.52 ms. These results show the benefits of tactical edge media processing and provide foundation for context-aware priority mechanisms in public-safety networks that integrate terrestrial and non-terrestrial components.

--------------------------------------------------

[4] (rt_087 | 2610.30015)
The study analyses the use of antennas in various internet-connected devices and the interference caused by metallic objects to their signal. To improve the network, multiple antennas with different polarisation are integrated. These experiments demonstrated a significant improvement in signal quality, with gains of up to 30GB in specific situations. Fitting a second antenna improves performance by up to 17 dB, based on PCB technology. All of the above suggests a close relationship between hardware and software, justifying the need for both to improve network stability for use in real-world environments.

--------------------------------------------------

[5] (rt_057 | 2611.08631)
Camera-based driving models build a dense bird's-eye-view (BEV) grid and spend the same computation on every cell of it, although the vehicle crosses only a few of those cells within the planning horizon. We present BEVCast, a two-stage BEV whose resolution adapts to the predicted driving path. A coarse pass encodes the whole scene on a grid of 4.096 m cells. A heat head trained on the recorded path assigns a heat to every coarse cell and the cells with the highest heat are refined into 1.024 m cells under a fixed budget. The remaining cells keep their coarse resolution, so the trajectory head reads a mixed-resolution BEV that still covers the entire scene and needs neither an HD map nor a navigation command. Against a full grid of 1.024 m cells with the same architecture and training recipe, BEVCast at a 10% refinement budget encodes 28% of the BEV queries and needs 66% of the BEV encoder time at batch size 1. On nuScenes its mean displacement error is 1.078 m against 1.149 m and its drivable area violation rate is 1.46% against 2.29%. On NAVSIM navtest its mean displacement error is 0.577 m against 0.598 m and its extended predictive driver model score is 82.51 against 81.28. Next to published planners BEVCast reaches a comparable error on both benchmarks without a navigation command. The resolution of a driving BEV can therefore follow the task instead of the grid.

--------------------------------------------------

[6] (rt_056 | 2611.12567)
Many tasks compose transformations by a known rule: chained rotations in robotics and navigation. Learned models relearn it, compounding error; classical estimators hardcode it and miss hardware deviations. We impose the known structure and learn only the remainder: an exact non-commutative group law sits in the node of a self-contained layer, its edges learn local increments, and a zero-initialized correction psi learns unknown deviations of the law. A pre-training criterion predicts exact representability. On fixed-angle SU(2) composition, a 54-parameter dense edge is the most accurate dense model, at 27x fewer parameters than the strongest fixed-matrix baseline, and lies 0.05 deg above the Bayes floor. On real robot arms, an isolated 306-parameter psi reaches 1.12-1.14 mm versus 1.29-1.33 mm for Gauss-Newton at N >= 60, and the recipe transfers across three vendors and two protocols. The recipe matches classical fits where the data obey the law and loses where the imposed law is not the task's.

--------------------------------------------------

[7] (rt_036 | 2610.05182)
Infrared image super-resolution is currently dominated by Mamba-based networks, which typically contain between 26 and 37 million parameters. This complexity hinders their deployment on aerial platforms or portable devices—environments where such technology is often essential. This paper introduces TIRMamba, a network designed for single-channel thermal imagery that utilizes only 896,000 to 910,000 parameters. A component termed "Thermal Prior Highway" computes the gradient, local contrast, and spectral features of the input. Through an adapter integrated into each residual group, this component modulates the behavior of a bidirectional state-space core—which employs shared weights—and governs a dual-scale detail branch. The architecture also incorporates a three-path reconstruction mechanism that adds the learned residual to a radiometric base obtained via bicubic interpolation. Given that standard benchmark datasets contain only 265 infrared images—making training and evaluation on full-sized images impractical—we adopted a data-reuse training strategy: an initial phase using the DIV2K dataset, followed by fine-tuning on 64-pixel patches extracted with equal probability from both infrared and natural image datasets. At a 4x scale factor, TIRMamba matches the performance of state-of-the-art methods trained under the same protocol, while requiring 29 to 40 times fewer parameters and exhibiting 0.4 to 2.8 times lower latency. At a 2x scale, it achieves the highest SSIM score. A variant, TIRMamba-Rad—which employs selectivity conditioned on prior information—improves metrics by 3 dB for raw thermal data, imagery captured by unmanned aerial vehicles (UAVs), and data from independent sensors, all at a 4x scale. The code is available at the following link: https://github.com/julian135707/TIRMamba

--------------------------------------------------

[8] (rt_030 | 2610.38121)
LOL-v1 and LOLv2-Real are the standard benchmarks for supervised low-light image enhancement; however, several prominent methods incorporate manual adjustments at test time within their evaluation scripts. Upon auditing both benchmarks using 18 publicly available checkpoints from eight different methods—applying a consistent metric implementation that reproduces each published figure within a margin of 0.09 dB—we discovered that the test labels differ from the training labels. First, the 15 test labels in LOL-v1 feature a different rendering; the same source captures appear in LOLv2 with identical luminance but 1.48x lower saturation, whereas the LOL-v1 training labels align with the rendering used in the official version of LOLv2. Second, the test set for LOLv2-Real (the contiguous block of images 690–789) contains labels that are 14% darker than the training labels (Cliff’s delta = -0.45, p < 1e-12), with no corresponding shift in the input statistics; this discrepancy allows methods trained without labels to achieve up to 5.5 dB higher performance on the test set than on the training set. Third, the 15 test inputs from LOL-v1 reappear in LOLv2-Real, with 12 of them included within its training set. The published test-time settings compensate for these variations: by separating components that can be calibrated using training data from those adjusted solely via test-set tuning, the "test-only residual" shows a median of 0.00 dB for LOLv2-Syn (where no such variations exist), but rises to 0.37 dB for LOL-v1 and 0.92 dB for LOLv2-Real (reaching as high as 3.9 dB). Given these shifts, selecting checkpoints based on the test set inflates the performance of a model trained from scratch by 0.9 to 2.0 dB; furthermore, removing these settings alters which method ranks first. Consequently, we provide concrete recommendations for reporting results.

--------------------------------------------------

[9] (rt_006 | 2611.11428)
Wireless foundation models learn reusable representations through different elements and, though they require a wide variety of environments, the wireless databases often carry an extensive amount of data in just a few places.
We present Channels4World, where people can access a dataset of 100.00 distinct RF outdoor scenes and their procedural method. Each of them is linked to a generated environment, RF material assigned to 32 base positions, and 16,348 UE positions; all of them aligned by their variable length from 3.5 to 7 GHz. Their identity is aligned so the geometry, placements, channel and cross-frequency interactions can be individually changed as needed.
The public release offers subsets at multiple scales and separate access to the different components, all kept together by a scene-level train-test. Therefore, Channels4World supports the development and evaluation of wireless models capable of generalising their functions through different environments and implementations.

--------------------------------------------------

[10] (rt_000 | 2611.12223)
Bit-Interleaved Coded Modulation is mostly used in wireless systems with multiple inputs and outputs. However, when taking only the soft information of every bit, the correlating information between the bits of each vector is lost. We demonstrate that this lost information limits the performance of the receptors, and although it can be mostly recovered through Iterative Detection and Decoding, it also makes it more complex.
To avoid it, we propose Correlation-Informed Logits, a soft output format that combines individual bit information with pairing logits.However, we will explain how to use these logits without IDD.

--------------------------------------------------

[11] (rt_019 | 2611.06053)
This article presents a new method to turn a millimetre-wave source with a Gaussian distribution into a flat-top by using a single curved surface. The lens is designed to operate at 100 GHz and is placed 52 mm in front of a conical horn antenna. The wave then goes from 11,9 mm to 400 mm. The lens profile is optimised through electromagnetic simulation to prove its success. And the sensitivity to frequency changes is investigated.

--------------------------------------------------

[12] (rt_021 | 2611.06050)
This paper proposes an analytical method to design single-sided plano-convex lenses that are capable of transforming a divergent Gaussian beam from monochromatic THz to flat-top in the Fresnel region. For this, two lenses are designed to work at 100 and 200 GHz that capture uniform waves with 400 and 500 mm radii at 3 and 4 meters. The stimuli demonstrate that the performance is similar to that of the conventional double-sided plano-convex lenses.
"""

raw_txt_path = "data/raw/respuestas_gold_real_sheets.txt"
os.makedirs("data/raw", exist_ok=True)
if os.path.exists("respuestas_gold_real_sheets.txt"):
    with open("respuestas_gold_real_sheets.txt", encoding="utf-8") as f:
        txt_content = f.read()
elif os.path.exists(raw_txt_path):
    with open(raw_txt_path, encoding="utf-8") as f:
        txt_content = f.read()
else:
    txt_content = RAW_TXT_FALLBACK

with open(raw_txt_path, "w", encoding="utf-8") as f:
    f.write(txt_content.strip() + "\n")

# 2. Parsear bloques [idx] (base_id | arxiv_id) y cruzar con roundtrip.jsonl
pattern = re.compile(
    r'\[(\d+)\]\s*\((rt_\d+)\s*\|\s*([^\)]+)\)\s*\n(.*?)(?=\n-{10,}|\Z)',
    re.DOTALL
)
matches = pattern.findall(txt_content)
assert len(matches) >= 12, f"Se esperaban al menos 12 bloques, se encontraron {len(matches)}"

with open("data/processed/roundtrip.jsonl", encoding="utf-8") as f:
    rt_by_base = {r["id"]: r for r in (json.loads(line) for line in f if line.strip())}

nlp = spacy.load("models/ner_hybrid_v1")
gold_records = []

for idx_str, base_id, arxiv_id, draft_raw in matches:
    idx = int(idx_str)
    base_rec = rt_by_base[base_id.strip()]
    assert base_rec["arxiv_id"] == arxiv_id.strip(), f"Discrepancia de arXiv ID en {base_id}"

    draft_clean = re.sub(r'\s+', ' ', draft_raw.strip())
    doc_a = nlp(base_rec["A"])
    doc_i = nlp(draft_clean)
    ents_a = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_a.ents]
    ents_i = [{"text": e.text.rstrip("."), "label": e.label_} for e in doc_i.ents]

    cov = compute_entity_coverage(ents_a, ents_i, draft_clean)
    lost = [e["text"] for e in ents_a if not entity_present(e, ents_i, draft_clean)]

    gold_records.append({
        "id": f"gold_{idx:02d}",
        "gold_idx": idx,
        "base_id": base_id.strip(),
        "arxiv_id": arxiv_id.strip(),
        "category": base_rec["category"],
        "topic": base_rec.get("topic", base_rec["category"]),
        "published_date": base_rec["published_date"],
        "split": "gold_real",
        "author_id": f"anon_member_{((idx - 1) // 3) + 1:02d}",
        "consent_recorded": True,
        "mode": "written",
        "A": base_rec["A"],
        "I": draft_clean,
        "entities_A": ents_a,
        "entities_I": ents_i,
        "survived_entity_cov": round(cov, 4),
        "forgotten_entities": lost
    })

out_gold_jsonl = "data/processed/gold_real.jsonl"
with open(out_gold_jsonl, "w", encoding="utf-8") as fout:
    for r in gold_records:
        fout.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"✔ Tarea T6 completada: {len(gold_records)} pares guardados en {out_gold_jsonl}")

# 3. Paperizar gold_real con Herramienta (P-PAP-v1) y Línea Base (P-BASE-v1)
paperized_gold_path = "data/processed/paperized_gold_real.jsonl"
if os.path.exists(paperized_gold_path):
    os.remove(paperized_gold_path)

eng = PaperizerEngine("configs/generacion.yaml")
run_paperization_pipeline(gold_records, paperized_gold_path, batch_size=6, engine=eng)
eng.close()

# 4. Evaluar métricas completas (BERTScore, ROUGE-L, Flesch, Entidades y Unidades) — Paso P8
df_gold = evaluate_all_systems(
    paperized_jsonl=paperized_gold_path,
    out_roundtrip_jsonl="results/gold_real_evaluated.jsonl",
    strict_keep_only=False
)

cols_p8 = [
    "id", "base_id", "arxiv_id", "category", "survived_entity_cov",
    "ent_cov_A", "ent_cov_I", "ent_prec_A", "unit_preserved",
    "bertscore_f1", "rougeL_f1", "flesch_I", "flesch_Aprime", "flesch_in_range"
]
df_gold_tool = df_gold[df_gold["prompt_id"] == "P-PAP-v1"][cols_p8].sort_values("id")
df_gold_base = df_gold[df_gold["prompt_id"] == "P-BASE-v1"][cols_p8].sort_values("id")
df_gold_tool.to_csv("results/tabla_gold_real_par_por_par.csv", index=False)

# 5. Calcular brecha descriptiva Sintético (roundtrip) vs Real (gold_real) sobre los MISMOS 12 abstracts
with open("results/roundtrip.jsonl", encoding="utf-8") as f:
    df_rt_all = pd.DataFrame([json.loads(line) for line in f if line.strip()])

df_rt_matched = df_rt_all[
    (df_rt_all["prompt_id"] == "P-PAP-v1") &
    (df_rt_all["arxiv_id"].isin(df_gold_tool["arxiv_id"]))
]

gap_metrics = ["ent_cov_A", "ent_cov_I", "ent_prec_A", "unit_preserved", "bertscore_f1", "rougeL_f1", "flesch_in_range"]
gap_rows = []
for m in gap_metrics:
    s_mean = float(df_rt_matched[m].astype(float).mean())
    r_tool = float(df_gold_tool[m].astype(float).mean())
    r_base = float(df_gold_base[m].astype(float).mean())
    gap_rows.append({
        "Metrica": m,
        "Sintetico_Herramienta_(mismos_12)": round(s_mean, 4),
        "GoldReal_Herramienta_(P-PAP-v1)": round(r_tool, 4),
        "GoldReal_LineaBase_(P-BASE-v1)": round(r_base, 4),
        "Brecha_(Real_menos_Sintetico)": round(r_tool - s_mean, 4)
    })

df_gap = pd.DataFrame(gap_rows)
df_gap.to_csv("results/tabla_brecha_sintetico_vs_real.csv", index=False)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("paperizar_evaluation")
with mlflow.start_run(run_name="gold_real_12_evaluation"):
    for _, row in df_gap.iterrows():
        mlflow.log_metric(f"gold_tool_{row['Metrica']}", row["GoldReal_Herramienta_(P-PAP-v1)"])
        mlflow.log_metric(f"gold_base_{row['Metrica']}", row["GoldReal_LineaBase_(P-BASE-v1)"])
        mlflow.log_metric(f"gap_real_vs_synth_{row['Metrica']}", row["Brecha_(Real_menos_Sintetico)"])

if os.path.exists("docs/EVALUACION.md"):
    with open("docs/EVALUACION.md", encoding="utf-8") as f:
        cur_md = f.read()
    sec_p8 = f"""\n\n## 4. Prueba con Borradores Reales (`gold_real`, N = 12 pares — Paso P8)
Reporte descriptivo par por par sobre 12 abstracts estratificados (2 por categoría de ingeniería), sin pruebas de significancia ni extrapolación:

{df_gold_tool.to_markdown(index=False)}

### Brecha Descriptiva: Sintético (`roundtrip`) vs. Humano (`gold_real`) sobre los MISMOS 12 Abstracts
{df_gap.to_markdown(index=False)}
"""
    if "## 4. Prueba con Borradores Reales" not in cur_md:
        with open("docs/EVALUACION.md", "w", encoding="utf-8") as f:
            f.write(cur_md.rstrip() + sec_p8 + "\n")

print("\n=== PASO P8: RESULTADOS PAR POR PAR EN GOLD_REAL (HERRAMIENTA P-PAP-v1) ===")
print(df_gold_tool.to_string(index=False))

print("\n=== BRECHA DESCRIPTIVA SINTÉTICO VS REAL (MISMOS 12 ABSTRACTS) ===")
print(df_gap.to_string(index=False))

Mounted at /content/drive
✔ Tarea T6 completada: 12 pares guardados en data/processed/gold_real.jsonl


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (22 outputs left out)

  0%|          | 0/2 [00:00<?, ?it/s]

done in 8.36 seconds, 4.30 sentences/sec


Métricas NER + Flesch + ROUGE:   0%|          | 0/36 [00:00<?, ?it/s]


=== PASO P8: RESULTADOS PAR POR PAR EN GOLD_REAL (HERRAMIENTA P-PAP-v1) ===
     id base_id   arxiv_id category  survived_entity_cov  ent_cov_A  ent_cov_I  ent_prec_A  unit_preserved  bertscore_f1  rougeL_f1  flesch_I  flesch_Aprime  flesch_in_range
gold_01  rt_096 2609.37711    cs.AR               1.0000     0.7333     0.7333      0.7857           False        0.4755     0.4451     30.02          13.39             True
gold_02  rt_063 2610.08688    cs.AR               0.8519     0.8148     0.9565      0.9524           False        0.6497     0.6388     35.06          33.71             True
gold_03  rt_084 2611.05165    cs.NI               1.0000     1.0000     1.0000      1.0000            True        0.6878     0.6949     11.98           4.55            False
gold_04  rt_087 2610.30015    cs.NI               0.5000     0.5000     0.6667      0.6667            True        0.4688     0.3905     22.88          -4.68            False
gold_05  rt_057 2611.08631    cs.RO               1.0

Qué incluye este evaluador HTML100% autocontenido y offline: Solo le envías el archivo .html por WhatsApp/Drive a tu compañero. Cada uno elige arriba si es Revisor 1 (rev1_40.csv) o Revisor 2 (rev2_40.csv).Calificación de 1 clic con autoguardado: Botones grandes del 1 al 5 para las tres dimensiones (Conserva significado, Registro académico y Fluidez) sobre Candidato X y Candidato Y, guardado automático en localStorage, barra de progreso 0 / 40, píldoras de salto rápido y tema oscuro.   Exportación exacta compatible con Colab: Al pulsar "Descargar CSV calificado", genera rev1_40.csv o rev2_40.csv con las columnas exactas (item_id, X_significado_1a5, X_registro_1a5, X_fluidez_1a5, Y_significado_1a5, Y_registro_1a5, Y_fluidez_1a5) y el escapado RFC-4180 listo para correr la Celda 8 en 05_evaluar.ipynb.   Celda para generar y descargar evaluador_ciego_40.html en tu Colab (05_evaluar.ipynb)

In [3]:
import os, json
import pandas as pd
from google.colab import drive, files

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/paperizar')

df_blind = pd.read_csv("results/hoja_revision_humana_40_ciega.csv").fillna("")
records_json = json.dumps(df_blind.to_dict(orient="records"), ensure_ascii=False).replace("</", "<\\/")

html_template = r"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Evaluación Humana a Ciegas (40 Pares) — Paperización</title>
  <style>
    :root {
      --bg: #f4f6f9;
      --surface: #ffffff;
      --surface-alt: #f8fafc;
      --border: #e2e8f0;
      --text: #0f172a;
      --text-muted: #64748b;
      --primary: #2563eb;
      --primary-hover: #1d4ed8;
      --success: #16a34a;
      --success-bg: #dcfce7;
      --success-border: #86efac;
      --candx-bg: #eff6ff;
      --candx-border: #bfdbfe;
      --candy-bg: #f5f3ff;
      --candy-border: #ddd6fe;
      --danger: #dc2626;
      --shadow: 0 1px 3px rgba(15, 23, 42, 0.08);
      --radius: 12px;
    }
    body.dark {
      --bg: #0b1120;
      --surface: #111827;
      --surface-alt: #1e293b;
      --border: #334155;
      --text: #f1f5f9;
      --text-muted: #94a3b8;
      --primary: #3b82f6;
      --primary-hover: #2563eb;
      --success: #22c55e;
      --success-bg: rgba(34, 197, 94, 0.16);
      --success-border: rgba(34, 197, 94, 0.45);
      --candx-bg: rgba(59, 130, 246, 0.12);
      --candx-border: rgba(59, 130, 246, 0.35);
      --candy-bg: rgba(139, 92, 246, 0.12);
      --candy-border: rgba(139, 92, 246, 0.35);
      --shadow: 0 2px 6px rgba(0, 0, 0, 0.35);
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Inter, sans-serif;
      background: var(--bg);
      color: var(--text);
      line-height: 1.55;
    }
    header {
      position: sticky; top: 0; z-index: 100;
      background: var(--surface); border-bottom: 1px solid var(--border);
      box-shadow: var(--shadow); padding: 12px 20px;
    }
    .header-inner { max-width: 1180px; margin: 0 auto; display: flex; flex-direction: column; gap: 10px; }
    .header-top { display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between; gap: 12px; }
    h1 { margin: 0; font-size: 1.12rem; font-weight: 700; }
    .subtitle { font-size: 0.8rem; color: var(--text-muted); }
    .toolbar { display: flex; flex-wrap: wrap; align-items: center; gap: 8px; }
    button, select {
      font: inherit; font-size: 0.84rem; border-radius: 8px;
      border: 1px solid var(--border); background: var(--surface-alt);
      color: var(--text); padding: 7px 12px; cursor: pointer; transition: 0.15s;
    }
    button:hover, select:hover { border-color: var(--primary); }
    button.primary { background: var(--primary); color: #fff; border-color: var(--primary); font-weight: 600; }
    button.primary:hover { background: var(--primary-hover); }
    button.danger { color: var(--danger); background: transparent; border-color: transparent; }
    button.danger:hover { background: rgba(220, 38, 38, 0.1); }
    .nav-bar { display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between; gap: 10px; font-size: 0.8rem; }
    .progress-wrap { display: flex; align-items: center; gap: 10px; min-width: 240px; }
    .progress-track { flex: 1; height: 8px; background: var(--border); border-radius: 999px; overflow: hidden; }
    .progress-fill { height: 100%; width: 0%; background: var(--success); transition: width 0.25s; }
    .jump-pills { display: flex; flex-wrap: wrap; gap: 4px; }
    .jump-pill {
      display: inline-flex; align-items: center; justify-content: center;
      min-width: 26px; height: 24px; padding: 0 5px; border-radius: 5px;
      font-size: 0.73rem; font-weight: 600; text-decoration: none;
      color: var(--text-muted); background: var(--surface-alt); border: 1px solid var(--border);
    }
    .jump-pill.done { background: var(--success-bg); color: var(--success); border-color: var(--success-border); }
    main { max-width: 1180px; margin: 20px auto 48px; padding: 0 20px; display: flex; flex-direction: column; gap: 20px; }
    .rubric-banner {
      background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius);
      padding: 12px 16px; font-size: 0.84rem; display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 12px;
    }
    .card {
      background: var(--surface); border: 1px solid var(--border);
      border-radius: var(--radius); box-shadow: var(--shadow); overflow: hidden;
      scroll-margin-top: 135px;
    }
    .card.completed { border-left: 4px solid var(--success); }
    .card-header {
      padding: 12px 18px; background: var(--surface-alt); border-bottom: 1px solid var(--border);
      display: flex; align-items: center; justify-content: space-between;
    }
    .idx-badge { background: var(--primary); color: #fff; font-weight: 700; font-size: 0.84rem; padding: 3px 10px; border-radius: 999px; }
    .status-badge { font-size: 0.76rem; font-weight: 600; padding: 3px 10px; border-radius: 999px; border: 1px solid var(--border); color: var(--text-muted); }
    .card.completed .status-badge { background: var(--success-bg); color: var(--success); border-color: var(--success-border); }
    .card-body { padding: 16px 18px; display: flex; flex-direction: column; gap: 14px; }
    .ref-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
    .text-box {
      background: var(--surface-alt); border: 1px solid var(--border);
      border-radius: 8px; padding: 12px 14px; font-size: 0.9rem; line-height: 1.6;
    }
    .box-label { font-size: 0.73rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.05em; color: var(--text-muted); margin-bottom: 5px; }
    .cand-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
    .cand-panel { border-radius: 10px; padding: 14px; display: flex; flex-direction: column; gap: 12px; }
    .cand-panel.cand-x { background: var(--candx-bg); border: 1px solid var(--candx-border); }
    .cand-panel.cand-y { background: var(--candy-bg); border: 1px solid var(--candy-border); }
    .cand-title { font-weight: 700; font-size: 0.95rem; display: flex; justify-content: space-between; }
    .cand-text { background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 12px; font-size: 0.92rem; line-height: 1.62; flex: 1; }
    .score-rows { display: flex; flex-direction: column; gap: 8px; }
    .score-row { display: flex; align-items: center; justify-content: space-between; gap: 8px; background: var(--surface); padding: 7px 10px; border-radius: 8px; border: 1px solid var(--border); }
    .score-row span { font-size: 0.83rem; font-weight: 600; }
    .likert { display: inline-flex; gap: 4px; }
    .likert button {
      width: 32px; height: 30px; padding: 0; font-weight: 700; font-size: 0.84rem;
      border-radius: 6px; background: var(--surface-alt); border: 1px solid var(--border);
    }
    .likert button.active { background: var(--primary); color: #fff; border-color: var(--primary); }
    .toast {
      position: fixed; bottom: 22px; right: 22px; background: #0f172a; color: #fff;
      padding: 10px 16px; border-radius: 8px; font-size: 0.85rem; opacity: 0;
      transform: translateY(10px); pointer-events: none; transition: 0.25s; z-index: 999;
    }
    .toast.show { opacity: 1; transform: translateY(0); }
    @media (max-width: 900px) { .ref-grid, .cand-grid { grid-template-columns: 1fr; } }
  </style>
</head>
<body>
  <header>
    <div class="header-inner">
      <div class="header-top">
        <div>
          <h1>Evaluación Humana a Ciegas — 40 Pares Estratificados (Paso P6)</h1>
          <div class="subtitle">Orden X/Y aleatorizado a ciegas · Escala Likert 1 (muy deficiente) a 5 (excelente)</div>
        </div>
        <div class="toolbar">
          <label style="font-size:0.84rem;font-weight:600;">
            Revisor:
            <select id="reviewerSelect">
              <option value="rev1">Revisor 1 (exporta rev1_40.csv)</option>
              <option value="rev2">Revisor 2 (exporta rev2_40.csv)</option>
            </select>
          </label>
          <button id="filterPendingBtn">Ver solo pendientes: OFF</button>
          <button id="themeBtn">Tema oscuro</button>
          <button id="exportCsvBtn" class="primary">Descargar CSV calificado</button>
          <button id="resetBtn" class="danger" title="Borrar calificaciones del revisor activo">Reiniciar</button>
        </div>
      </div>
      <div class="nav-bar">
        <div class="progress-wrap">
          <strong id="progressText">0 / 40 completados</strong>
          <div class="progress-track"><div id="progressFill" class="progress-fill"></div></div>
        </div>
        <div id="jumpPills" class="jump-pills"></div>
      </div>
    </div>
  </header>

  <main>
    <section class="rubric-banner">
      <div><strong>1. Conserva significado (1–5):</strong> ¿Mantiene exactamente los hechos, cifras, unidades y siglas del borrador/original sin inventar ni omitir?</div>
      <div><strong>2. Registro académico (1–5):</strong> ¿Usa tono formal, objetivo e impersonal propio de un abstract de ingeniería (sin jerga ni coloquialismos)?</div>
      <div><strong>3. Fluidez (1–5):</strong> ¿Es gramaticalmente correcto, cohesivo y natural de leer en inglés?</div>
    </section>
    <div id="cardsContainer"></div>
  </main>

  <div id="toast" class="toast"></div>

  <script>
    const ITEMS = __RECORDS_JSON__;
    const FIELDS = [
      "X_significado_1a5", "X_registro_1a5", "X_fluidez_1a5",
      "Y_significado_1a5", "Y_registro_1a5", "Y_fluidez_1a5"
    ];
    const THEME_KEY = "paperizar_blind40_theme";
    let currentReviewer = localStorage.getItem("paperizar_blind40_rev") || "rev1";
    let onlyPending = false;
    let state = loadState(currentReviewer);

    function storageKey(rev) { return `paperizar_blind40_scores_${rev}_v1`; }

    function loadState(rev) {
      try {
        const raw = localStorage.getItem(storageKey(rev));
        return raw ? JSON.parse(raw) : {};
      } catch (e) { return {}; }
    }

    function saveState() {
      localStorage.setItem(storageKey(currentReviewer), JSON.stringify(state));
    }

    function showToast(msg) {
      const t = document.getElementById("toast");
      t.textContent = msg; t.classList.add("show");
      clearTimeout(t._tm); t._tm = setTimeout(() => t.classList.remove("show"), 2600);
    }

    function esc(s) {
      return String(s || "")
        .replace(/&/g, "&amp;").replace(/</g, "&lt;")
        .replace(/>/g, "&gt;").replace(/"/g, "&quot;");
    }

    function isItemComplete(itemId) {
      const s = state[itemId] || {};
      return FIELDS.every(f => Number.isInteger(s[f]) && s[f] >= 1 && s[f] <= 5);
    }

    function renderScoreRow(itemId, fieldKey, labelText) {
      const curVal = (state[itemId] || {})[fieldKey] || null;
      const btns = [1, 2, 3, 4, 5].map(v =>
        `<button type="button" class="${curVal === v ? 'active' : ''}" data-item="${itemId}" data-field="${fieldKey}" data-val="${v}">${v}</button>`
      ).join("");
      return `<div class="score-row"><span>${labelText}</span><div class="likert">${btns}</div></div>`;
    }

    function buildCards() {
      const container = document.getElementById("cardsContainer");
      const pills = document.getElementById("jumpPills");
      container.innerHTML = ""; pills.innerHTML = "";

      ITEMS.forEach((item, i) => {
        const id = item.item_id;
        const num = i + 1;

        const pill = document.createElement("a");
        pill.href = `#card-${id}`;
        pill.id = `pill-${id}`;
        pill.className = "jump-pill";
        pill.textContent = num;
        pills.appendChild(pill);

        const card = document.createElement("article");
        card.className = "card";
        card.id = `card-${id}`;
        card.innerHTML = `
          <div class="card-header">
            <span class="idx-badge">${esc(id)} (${num} / ${ITEMS.length})</span>
            <span class="status-badge" id="status-${id}">Pendiente</span>
          </div>
          <div class="card-body">
            <div class="ref-grid">
              <div>
                <div class="box-label">Abstract Original de Referencia (A)</div>
                <div class="text-box">${esc(item.original_A)}</div>
              </div>
              <div>
                <div class="box-label">Borrador Informal de Entrada (I)</div>
                <div class="text-box">${esc(item.draft_I)}</div>
              </div>
            </div>
            <div class="cand-grid">
              <div class="cand-panel cand-x">
                <div class="cand-title"><span>Candidato X (ciego)</span></div>
                <div class="cand-text">${esc(item.candidate_X)}</div>
                <div class="score-rows">
                  ${renderScoreRow(id, "X_significado_1a5", "Conserva significado")}
                  ${renderScoreRow(id, "X_registro_1a5", "Registro académico")}
                  ${renderScoreRow(id, "X_fluidez_1a5", "Fluidez")}
                </div>
              </div>
              <div class="cand-panel cand-y">
                <div class="cand-title"><span>Candidato Y (ciego)</span></div>
                <div class="cand-text">${esc(item.candidate_Y)}</div>
                <div class="score-rows">
                  ${renderScoreRow(id, "Y_significado_1a5", "Conserva significado")}
                  ${renderScoreRow(id, "Y_registro_1a5", "Registro académico")}
                  ${renderScoreRow(id, "Y_fluidez_1a5", "Fluidez")}
                </div>
              </div>
            </div>
          </div>
        `;
        container.appendChild(card);
      });

      refreshUI();
    }

    function refreshUI() {
      let doneCount = 0;
      ITEMS.forEach(item => {
        const id = item.item_id;
        const done = isItemComplete(id);
        if (done) doneCount++;
        const card = document.getElementById(`card-${id}`);
        const st = document.getElementById(`status-${id}`);
        const pill = document.getElementById(`pill-${id}`);
        if (card) {
          card.classList.toggle("completed", done);
          card.style.display = (onlyPending && done) ? "none" : "";
        }
        if (st) st.textContent = done ? "Completado ✓" : "Pendiente";
        if (pill) pill.classList.toggle("done", done);
      });
      document.getElementById("progressText").textContent = `${doneCount} / ${ITEMS.length} completados`;
      document.getElementById("progressFill").style.width = `${Math.round((doneCount / ITEMS.length) * 100)}%`;
    }

    document.getElementById("cardsContainer").addEventListener("click", e => {
      const btn = e.target.closest("button[data-item]");
      if (!btn) return;
      const itemId = btn.getAttribute("data-item");
      const field = btn.getAttribute("data-field");
      const val = parseInt(btn.getAttribute("data-val"), 10);

      if (!state[itemId]) state[itemId] = {};
      state[itemId][field] = val;
      saveState();

      btn.parentElement.querySelectorAll("button").forEach(b => {
        b.classList.toggle("active", parseInt(b.getAttribute("data-val"), 10) === val);
      });
      refreshUI();
    });

    function csvEscape(val) {
      const s = String(val ?? "");
      if (/[",\r\n]/.test(s)) return `"${s.replace(/"/g, '""')}"`;
      return s;
    }

    document.getElementById("exportCsvBtn").addEventListener("click", () => {
      const missing = ITEMS.filter(it => !isItemComplete(it.item_id)).map(it => it.item_id);
      if (missing.length > 0) {
        const ok = confirm(`Aún faltan ${missing.length} pares por completar (${missing.slice(0, 5).join(", ")}${missing.length > 5 ? "..." : ""}). ¿Descargar de todos modos?`);
        if (!ok) return;
      }

      const headers = [
        "item_id", "original_A", "draft_I", "candidate_X",
        "X_significado_1a5", "X_registro_1a5", "X_fluidez_1a5",
        "candidate_Y", "Y_significado_1a5", "Y_registro_1a5", "Y_fluidez_1a5"
      ];
      const lines = [headers.join(",")];
      ITEMS.forEach(it => {
        const s = state[it.item_id] || {};
        const row = [
          it.item_id, it.original_A, it.draft_I, it.candidate_X,
          s.X_significado_1a5 ?? "", s.X_registro_1a5 ?? "", s.X_fluidez_1a5 ?? "",
          it.candidate_Y, s.Y_significado_1a5 ?? "", s.Y_registro_1a5 ?? "", s.Y_fluidez_1a5 ?? ""
        ].map(csvEscape);
        lines.push(row.join(","));
      });

      const fname = `${currentReviewer}_40.csv`;
      const blob = new Blob(["\uFEFF" + lines.join("\n") + "\n"], { type: "text/csv;charset=utf-8" });
      const a = document.createElement("a");
      a.href = URL.createObjectURL(blob);
      a.download = fname;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      showToast(`Descargado: ${fname}`);
    });

    document.getElementById("reviewerSelect").value = currentReviewer;
    document.getElementById("reviewerSelect").addEventListener("change", e => {
      currentReviewer = e.target.value;
      localStorage.setItem("paperizar_blind40_rev", currentReviewer);
      state = loadState(currentReviewer);
      buildCards();
      showToast(`Cambiado a ${currentReviewer === 'rev1' ? 'Revisor 1' : 'Revisor 2'}`);
    });

    document.getElementById("filterPendingBtn").addEventListener("click", e => {
      onlyPending = !onlyPending;
      e.target.textContent = `Ver solo pendientes: ${onlyPending ? "ON" : "OFF"}`;
      refreshUI();
    });

    document.getElementById("resetBtn").addEventListener("click", () => {
      if (!confirm(`¿Seguro que deseas borrar todas las calificaciones guardadas de ${currentReviewer}?`)) return;
      state = {}; saveState(); buildCards();
      showToast("Calificaciones reiniciadas.");
    });

    function setDark(d) {
      document.body.classList.toggle("dark", d);
      document.getElementById("themeBtn").textContent = d ? "Tema claro" : "Tema oscuro";
      localStorage.setItem(THEME_KEY, d ? "dark" : "light");
    }
    document.getElementById("themeBtn").addEventListener("click", () => setDark(!document.body.classList.contains("dark")));
    if (localStorage.getItem(THEME_KEY) === "dark") setDark(true);

    buildCards();
  </script>
</body>
</html>
"""

final_html = html_template.replace("__RECORDS_JSON__", records_json)
out_html_path = "results/evaluador_ciego_40.html"
with open(out_html_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"✔ Evaluador generado en Drive: {out_html_path}")
files.download(out_html_path)

✔ Evaluador generado en Drive: results/evaluador_ciego_40.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>